# Train โมเดลเมล็ดโรบัสต้า 7 คลาสบน Colab

ก่อนเริ่ม:
1. **Runtime → Change runtime type → T4 GPU**
2. อัปโหลดไฟล์จากเครื่องขึ้น Google Drive ตามโครงนี้ (ชื่อโฟลเดอร์เปลี่ยนได้ที่ cell ตั้งค่า)
```
MyDrive/TESA/vision/
├── scripts/        <- ทั้งโฟลเดอร์ vision/scripts
└── datasets.zip    <- vision/datasets.zip
```
โค้ดและ dataset จะถูก copy มาไว้ที่ `/content/vision` (ดิสก์เครื่อง Colab เร็วกว่า Drive) ส่วนผล train เขียนลง Drive โดยตรง ถ้า Colab หลุด ผลไม่หาย

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## ตั้งค่า

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/TESA/vision'   # ที่เก็บ scripts/ และ datasets.zip
RUNS_DIR  = f'{DRIVE_DIR}/runs'                     # ผล train ทุก run อยู่ที่นี่
WORK      = '/content/vision'

## เตรียมโค้ด + dataset + ไลบรารี (รันใหม่ทุกครั้งที่เปิด runtime ใหม่)

In [ ]:
!mkdir -p {WORK} {RUNS_DIR}
!rm -rf {WORK}/scripts && cp -r "{DRIVE_DIR}/scripts" {WORK}/
!cd {WORK} && unzip -qo "{DRIVE_DIR}/datasets.zip" -x '__MACOSX/*' '*.DS_Store'
!ls {WORK}/datasets/robusta-7cls
!pip -q install ultralytics==8.4.184

## 1) Smoke test: เช็กว่า pipeline ไม่พังก่อน train จริง (1 epoch ใช้ไม่กี่นาที)
ถ้า cell นี้จบและพิมพ์บรรทัด `TEST ...` ได้ แปลว่าพร้อม train จริง

In [ ]:
%cd {WORK}
!python scripts/train_segmentation.py --name smoke_seg --epochs 1 --fraction 0.2 --workers 2 --project "{RUNS_DIR}"

## 2) Train ตัวหลัก: Segmentation (ใช้ทำทั้ง seg, กรอบ และ crop ให้ classifier)
- `--name` ห้ามซ้ำกับ run เดิม (สคริปต์จะ error กันผลเก่าโดนทับ) ให้เพิ่มเลขเวอร์ชันทุกครั้ง
- `--export ncnn` ได้ไฟล์สำหรับ Raspberry Pi ทันทีหลัง train เสร็จ
- ทดลองรุ่นใหญ่ขึ้นได้ด้วย `--model yolo26s-seg.pt` (แม่นขึ้น แต่ช้าลงบน Pi)

In [ ]:
!python scripts/train_segmentation.py --name seg_y26n_v1 --workers 2 --project "{RUNS_DIR}" --export ncnn

## 3) (ตัวเทียบ) Detection

In [ ]:
!python scripts/train_detection.py --name det_y26n_v1 --workers 2 --project "{RUNS_DIR}" --export ncnn

## 4) Classification (ทีละเม็ด สำหรับงาน Day 1)

In [ ]:
!python scripts/train_classification.py --name cls_y26n_v1 --workers 2 --project "{RUNS_DIR}" --export ncnn

## ดูผล
เปลี่ยน `RUN` เป็นชื่อ run ที่อยากดู กราฟ loss/mAP อยู่ใน `results.png` และผลบนชุด test อยู่ในโฟลเดอร์ `<RUN>_test`

In [ ]:
from IPython.display import Image, display
import os
RUN = 'seg_y26n_v1'
for f in [f'{RUNS_DIR}/{RUN}/results.png',
          f'{RUNS_DIR}/{RUN}_test/confusion_matrix_normalized.png']:
    if os.path.exists(f):
        print(f); display(Image(f, width=900))

## Colab หลุดกลางทาง: train ต่อจาก epoch ล่าสุด
ใส่ชื่อ run ที่ค้างไว้ แล้วรัน (ต้องรัน cell เตรียมโค้ดก่อน) จบแล้วค่อยรัน test/export เองด้วย cell ถัดไป

In [ ]:
from ultralytics import YOLO
RUN = 'seg_y26n_v1'
YOLO(f'{RUNS_DIR}/{RUN}/weights/last.pt').train(resume=True)

In [ ]:
from ultralytics import YOLO
RUN = 'seg_y26n_v1'
m = YOLO(f'{RUNS_DIR}/{RUN}/weights/best.pt')
r = m.val(split='test', project=RUNS_DIR, name=f'{RUN}_test', exist_ok=True)
print(m.export(format='ncnn', imgsz=640))   # cls ใช้ imgsz=224